# Task 3: Image Captioning AI (CNN-LSTM & Transformer)
### CODSOFT AI Internship
**Author:** Pratik  
**Domain:** Artificial Intelligence  

---

## 📖 Introduction to Image Captioning
**Image Captioning** is the task of generating a textual description for a given image. It combines concepts from:
1. **Computer Vision (CV)**: To understand the visual context of the image.
2. **Natural Language Processing (NLP)**: To generate a syntactically and semantically correct sentence describing that context.

In this notebook, we explore the two primary paradigms for this task:
- **Classic Paradigm (Encoder-Decoder / CNN-RNN)**: A pre-trained CNN (like ResNet or VGG) extracts spatial features, which are then passed to an LSTM/RNN sequence generator.
- **Modern Paradigm (Vision-Language Transformers)**: End-to-end transformers (like BLIP or ViT-GPT2) that use attention mechanisms to map visual patches directly to text tokens.

## 📐 Part 1: Classic Encoder-Decoder CNN-LSTM Architecture

### Pipeline Overview
```text
  [Input Image] 
        │
  [ResNet-50 CNN Encoder] (Remove classification head)
        │
  [Feature Vector] (e.g. size 2048) -> projected to Embed Size (e.g. 256)
        │
  [LSTM Decoder] (First hidden state initialized with visual feature)
        │
  [Word Predictions] (<start> -> "a" -> "dog" -> "running" -> <end>)
```

Let's import PyTorch and define this classic architecture.

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models

# CNN Feature Extractor (Encoder)
class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        modules = list(resnet.children())[:-1] # Remove classification layer
        self.resnet = nn.Sequential(*modules)
        self.embed = nn.Linear(resnet.fc.in_features, embed_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, images):
        features = self.resnet(images)
        features = features.view(features.size(0), -1)
        features = self.dropout(self.relu(self.embed(features)))
        return features

# Sequence Generator (Decoder)
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(DecoderRNN, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, captions):
        embeddings = self.dropout(self.embed(captions[:, :-1]))
        inputs = torch.cat((features.unsqueeze(1), embeddings), dim=1)
        hiddens, _ = self.lstm(inputs)
        outputs = self.linear(hiddens)
        return outputs

## 🏃‍♂️ Part 2: Training Pipeline Concept
To train a custom encoder-decoder model, you would typically follow these steps:
1. **Dataset**: Use a benchmark dataset like **Flickr8k** or **COCO** containing thousands of images with 5 reference captions each.
2. **Vocabulary Build**: Tokenize all captions, discard words below a frequency threshold, and map tokens to numerical indices.
3. **Preprocessing**: Resize images to `224x224`, normalize using ImageNet channel statistics, and pad caption sequences to equal lengths.
4. **Training Loop**: Use **Teacher Forcing** (passing the actual target word as the next step's input regardless of LSTM's prediction) with **Cross Entropy Loss** to train the model weights.

## 🤖 Part 3: Modern Pre-trained Transformers (BLIP Inference)
Instead of training a CNN-LSTM from scratch, we can leverage state-of-the-art **pretrained vision-language models** like Salesforce's **BLIP (Bootstrapped Language-Image Pre-training)**. Let's see how simple it is to generate captions using Hugging Face's `transformers` library!

In [2]:
# The code below shows how to fetch and run BLIP on any custom image
"""
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

# Load processor and model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Open your image
image = Image.open("test_image.jpg").convert("RGB")

# Generate caption
inputs = processor(image, return_tensors="pt")
out = model.generate(**inputs, max_new_tokens=40)
caption = processor.decode(out[0], skip_special_tokens=True)

print("Caption:", caption.capitalize())
"""
print("Hugging Face BLIP pipeline ready for production deployment!")

## 🎓 Conclusion
1. **Encoder-Decoder (CNN + LSTM)** is an excellent pedagogical baseline. It highlights how modular components of CV and NLP can connect together through simple projection layers.
2. **Transformers (BLIP)** represent the industry standard. They replace recurrence with global attention, allowing the model to focus on specific sections of the image while generating each corresponding word, resulting in extremely rich, detailed captions.